# EduPulse — 탐색적 데이터 분석 (EDA)

합성 데이터의 구조, 분포, 계절성 패턴을 탐색합니다.

**분석 항목:**
1. 데이터 로딩 및 기본 통계
2. 분야별 등록 인원 분포
3. 시계열 트렌드 및 계절성
4. 외부 데이터 (검색 트렌드, 채용 공고) 패턴
5. 변수 간 상관관계

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. 데이터 로딩

In [ ]:
# 합성 데이터가 없으면 생성
import os
if not os.path.exists('../edupulse/data/raw/internal/enrollment_history.csv'):
    print('합성 데이터 생성 중...')
    from edupulse.data.generators.run_all import run
    run()
    print('완료.')

enrollment = pd.read_csv('../edupulse/data/raw/internal/enrollment_history.csv')
search = pd.read_csv('../edupulse/data/raw/external/search_trends.csv')
jobs = pd.read_csv('../edupulse/data/raw/external/job_postings.csv')

print(f'수강 이력: {enrollment.shape}')
print(f'검색 트렌드: {search.shape}')
print(f'채용 공고: {jobs.shape}')

## 2. 기본 통계

In [ ]:
print('=== 수강 이력 ===')
display(enrollment.describe())
print('\n결측치:')
display(enrollment.isnull().sum())
print('\n분야별 건수:')
display(enrollment['field'].value_counts())

In [ ]:
print('=== 검색 트렌드 ===')
display(search.describe())
print('\n=== 채용 공고 ===')
display(jobs.describe())

## 3. 분야별 등록 인원 분포

In [ ]:
fields = enrollment['field'].unique()

fig, axes = plt.subplots(1, len(fields), figsize=(4 * len(fields), 4), sharey=True)
for ax, field in zip(axes, fields):
    data = enrollment[enrollment['field'] == field]['enrollment_count']
    ax.hist(data, bins=20, edgecolor='black', alpha=0.7)
    ax.set_title(f'{field}\n(평균: {data.mean():.1f})')
    ax.set_xlabel('등록 인원')
axes[0].set_ylabel('빈도')
plt.suptitle('분야별 등록 인원 분포', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 수요 등급 분포
from edupulse.constants import classify_demand

enrollment['demand_tier'] = enrollment['enrollment_count'].apply(classify_demand)
tier_counts = enrollment.groupby(['field', 'demand_tier']).size().unstack(fill_value=0)

tier_counts.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='RdYlGn_r')
plt.title('분야별 수요 등급 분포')
plt.xlabel('분야')
plt.ylabel('건수')
plt.xticks(rotation=0)
plt.legend(title='수요 등급')
plt.tight_layout()
plt.show()

## 4. 시계열 트렌드

In [ ]:
enrollment['date'] = pd.to_datetime(enrollment['date'])

fig, axes = plt.subplots(len(fields), 1, figsize=(14, 3 * len(fields)), sharex=True)
for ax, field in zip(axes, fields):
    field_data = enrollment[enrollment['field'] == field].sort_values('date')
    ax.plot(field_data['date'], field_data['enrollment_count'], alpha=0.6, linewidth=0.8)
    # 4주 이동 평균
    rolling = field_data.set_index('date')['enrollment_count'].rolling('28D').mean()
    ax.plot(rolling.index, rolling.values, color='red', linewidth=1.5, label='4주 이동평균')
    ax.set_ylabel('등록 인원')
    ax.set_title(f'{field}')
    ax.legend(loc='upper left')

plt.suptitle('분야별 주간 등록 인원 추이', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 월별 계절성 패턴
enrollment['month'] = enrollment['date'].dt.month

fig, axes = plt.subplots(1, len(fields), figsize=(4 * len(fields), 4), sharey=True)
for ax, field in zip(axes, fields):
    field_data = enrollment[enrollment['field'] == field]
    monthly = field_data.groupby('month')['enrollment_count'].mean()
    ax.bar(monthly.index, monthly.values, color='steelblue', alpha=0.8)
    ax.set_title(f'{field}')
    ax.set_xlabel('월')
    ax.set_xticks(range(1, 13))
axes[0].set_ylabel('평균 등록 인원')
plt.suptitle('월별 평균 등록 인원 (계절성)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. 외부 데이터 패턴

In [ ]:
search['date'] = pd.to_datetime(search['date'])
jobs['date'] = pd.to_datetime(jobs['date'])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for field in fields:
    s = search[search['field'] == field].sort_values('date')
    axes[0].plot(s['date'], s['search_volume'], alpha=0.7, label=field)
axes[0].set_title('분야별 검색 트렌드')
axes[0].set_ylabel('검색량')
axes[0].legend()

for field in fields:
    j = jobs[jobs['field'] == field].sort_values('date')
    axes[1].plot(j['date'], j['job_count'], alpha=0.7, label=field)
axes[1].set_title('분야별 채용 공고 추이')
axes[1].set_ylabel('채용 공고 수')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. 변수 간 상관관계

In [ ]:
# 병합 데이터로 상관관계 분석
from edupulse.preprocessing.merger import build_training_dataset

training = build_training_dataset()
print(f'학습 데이터셋: {training.shape}')
display(training.head())

In [ ]:
numeric_cols = training.select_dtypes(include='number').columns.tolist()
corr = training[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(numeric_cols, fontsize=9)

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)

plt.colorbar(im)
plt.title('피처 상관관계 히트맵')
plt.tight_layout()
plt.show()

In [ ]:
# enrollment_count와의 상관관계 순위
target_corr = corr['enrollment_count'].drop('enrollment_count').sort_values(key=abs, ascending=False)
print('enrollment_count와의 상관관계 (절대값 순):')
for col, val in target_corr.items():
    print(f'  {col:25s}: {val:+.4f}')